# Day 04 下午：电商用户行为数据清洗项目

**项目数据：** E Commerce Dataset.xlsx（E Comm 工作表）  
**项目目标：** 将上午学习的处理方法固化为可复用的数据清洗流程，并交付可供第五天分析使用的数据文件。

## 最终交付物

运行本 Notebook 后，应在 output/day04_project/ 中生成：

1. ecommerce_customer_cleaned.csv：清洗后的用户数据；
2. data_quality_before.csv：清洗前质量报告；
3. data_quality_after.csv：清洗后质量报告；
4. cleaning_log.csv：数据处理日志。

## 个人GitHub项目说明

- 每名学生独立完成本Notebook；
- 输入文件固定为`data/E Commerce Dataset.xlsx`；
- 输出固定写入`output/day04_project/`；
- 不要提交教师演示Notebook或教师参考答案；
- 完成后重启内核并从头运行，再推送到个人GitHub仓库。

## 项目规则

- 原始数据只读，不覆盖；
- 清洗函数接收 DataFrame，返回清洗结果与处理日志；
- 处理规则必须可解释；
- 不使用 Churn 分组填补特征，避免将目标变量信息带入特征处理；
- 发现候选异常值后，先记录和判断，不盲目删除。

---
## 1. 项目初始化与数据读取

In [11]:
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd


pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")


def find_project_root(start=None):
    """从当前目录向上寻找种子项目根目录。"""
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "E Commerce Dataset.xlsx").exists():
            return candidate
    raise FileNotFoundError("未找到data/E Commerce Dataset.xlsx")


PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "data" / "E Commerce Dataset.xlsx"
OUTPUT_DIR = PROJECT_ROOT / "output" / "day04_project"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


raw_df = pd.read_excel(DATA_PATH, sheet_name="E Comm")

print(f"原始数据：{DATA_PATH}")
print(f"项目输出目录：{OUTPUT_DIR}")
print(f"原始数据形状：{raw_df.shape}")
raw_df.head()

原始数据：C:\Users\Lenovo\PyCharmMiscProject\ecommerce-user-analysis-24012399\data\E Commerce Dataset.xlsx
项目输出目录：C:\Users\Lenovo\PyCharmMiscProject\ecommerce-user-analysis-24012399\output\day04_project
原始数据形状：(5630, 20)


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93
1,50002,1,NaN,Phone,1,8.00,UPI,Male,3.00,4,Mobile,3,Single,7,1,15.00,0.00,1.00,0.00,120.90
2,50003,1,NaN,Phone,1,30.00,Debit Card,Male,2.00,4,Mobile,3,Single,6,1,14.00,0.00,1.00,3.00,120.28
3,50004,1,0.00,Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07
4,50005,1,0.00,Phone,1,12.00,CC,Male,NaN,3,Mobile,5,Single,3,0,11.00,1.00,1.00,3.00,129.60


### 任务 1：确认项目对象

请回答：

1. 每条记录代表什么？
2. 项目的目标变量是哪一列？
3. 为什么 CustomerID 不应作为普通连续数值参与后续分析？

In [ ]:
# 在此写下你的答案：
# 1.每条记录代表一位电商平台独立用户的全量画像与行为样本，一行对应单个用户。
# 2.项目目标变量为 Churn 字段，1=用户流失，0=用户留存，是二分类预测标签。
# 3.CustomerID 是用户唯一标识编号，仅用于区分个体，数字大小无业务数值含义，不存在大小、大小排序相关性；直接作为数值特征会造成模型严重过拟合，模型记忆每个ID对应流失标签，无法泛化到新用户；若ID按注册顺序递增，会引入时间维度数据泄露，干扰真实特征相关性；标识符类字段不具备统计预测价值，不能用于建模，如需使用需聚合衍生特征（如同批次注册密度）。

---
## 2. 构建数据质量报告

质量报告至少应包含字段类型、缺失数量、缺失比例和唯一值数量。它用于对比清洗前后数据质量。

In [12]:
def build_quality_report(data):
    """返回字段级数据质量报告。"""
    # TODO：返回一个 DataFrame，至少包含：
    # 数据类型、缺失数量、缺失比例(%)、唯一值数量
    missing_count = data.isna().sum()
    missing_pct = (data.isna().mean() * 100).round(2)
    dtype_series = data.dtypes
    unique_count = data.nunique()

    quality_report = pd.DataFrame({
        "字段类型": dtype_series,
        "缺失数量": missing_count,
        "缺失比例(%)": missing_pct,
        "唯一值数量": unique_count
    })
    quality_report = quality_report.sort_values("缺失数量", ascending=False)
    return quality_report

# TODO：生成清洗前质量报告
quality_before = build_quality_report(raw_df)
display(quality_before)

,字段类型,缺失数量,缺失比例(%),唯一值数量
DaySinceLastOrder,float64,307,5.45,22
OrderAmountHikeFromlastYear,float64,265,4.71,16
Tenure,float64,264,4.69,36
OrderCount,float64,258,4.58,16
CouponUsed,float64,256,4.55,17
HourSpendOnApp,float64,255,4.53,6
WarehouseToHome,float64,251,4.46,34
CustomerID,int64,0,0.00,5630
PreferredLoginDevice,str,0,0.00,3
Churn,int64,0,0.00,2


### 任务 2：完成初始审计

除字段级质量报告外，请输出：

- 原始数据的完全重复行数；
- CustomerID 重复数量；
- Churn 的频数和流失率；
- 主要类别字段的频数。

In [13]:
# TODO：完成项目初始审计
# 1. 完全重复行数量
dup_total = raw_df.duplicated().sum()
print("完全重复行：", dup_total)

# 2. CustomerID重复数量
dup_cid = raw_df["CustomerID"].duplicated().sum()
print("CustomerID 重复数量：", dup_cid)

# 3. Churn流失标签频数 & 流失率
print("\nChurn 标签分布：")
print(raw_df["Churn"].value_counts())
churn_rate = raw_df["Churn"].mean() * 100
print(f"流失率：{churn_rate:.2f}%")

# 4. 核心类别字段取值频数
cat_cols = ["PreferredLoginDevice", "PreferredPaymentMode", "PreferedOrderCat"]
for col in cat_cols:
    print(f"\n===== {col} 取值分布 =====")
    print(raw_df[col].value_counts())

完全重复行： 0
CustomerID 重复数量： 0

Churn 标签分布：
Churn
0    4682
1     948
Name: count, dtype: int64
流失率：16.84%

===== PreferredLoginDevice 取值分布 =====
PreferredLoginDevice
Mobile Phone    2765
Computer        1634
Phone           1231
Name: count, dtype: int64

===== PreferredPaymentMode 取值分布 =====
PreferredPaymentMode
Debit Card          2314
Credit Card         1501
E wallet             614
UPI                  414
COD                  365
CC                   273
Cash on Delivery     149
Name: count, dtype: int64

===== PreferedOrderCat 取值分布 =====
PreferedOrderCat
Laptop & Accessory    2050
Mobile Phone          1271
Fashion                826
Mobile                 809
Grocery                410
Others                 264
Name: count, dtype: int64


---
## 3. 定义清洗规则

本项目采用以下规则：

| 问题 | 处理规则 | 理由 |
|---|---|---|
| 数值字段缺失 | 使用总体中位数填补 | 稳健且不将缺失误解为 0 |
| Phone / Mobile Phone | 统一为 Mobile Phone | 同一业务类别 |
| COD / Cash on Delivery | 统一为 Cash on Delivery | 同一业务类别 |
| CC / Credit Card | 统一为 Credit Card | 同一业务类别 |
| Mobile / Mobile Phone | 统一为 Mobile Phone | 同一业务类别 |
| 完全重复行 | 若存在则删除 | 完全相同的记录不增加信息 |
| 业务不合规值 | 记录并复核 | 本数据不应仅凭 IQR 直接删除 |

注意：不按 Churn 分组填补缺失值。

In [14]:
NUMERIC_MISSING_COLS = [
    "Tenure",
    "WarehouseToHome",
    "HourSpendOnApp",
    "OrderAmountHikeFromlastYear",
    "CouponUsed",
    "OrderCount",
    "DaySinceLastOrder",
]

CATEGORY_MAPPINGS = {
    "PreferredLoginDevice": {
        "Phone": "Mobile Phone"
    },
    "PreferredPaymentMode": {
        "COD": "Cash on Delivery",
        "CC": "Credit Card"
    },
    "PreferedOrderCat": {
        "Mobile": "Mobile Phone"
    }
}

---
## 4. 编写可复用清洗函数

函数要求：

- 不直接修改传入的原始 DataFrame；
- 返回 cleaned_df 和 cleaning_log；
- 日志至少包含处理步骤、处理规则、处理前记录数、处理后记录数、影响记录数；
- 完成重复值处理、缺失值处理、类别标准化和必要的数据类型转换。

In [15]:
def clean_ecommerce_data(data):
    """
    清洗电商用户行为数据。

    参数：
        data: 原始用户行为 DataFrame

    返回：
        cleaned_df: 清洗后的 DataFrame
        cleaning_log: 处理日志 DataFrame
    """
    # TODO：复制数据，避免覆盖原始数据
    df = data.copy()
    # TODO：创建日志列表 logs
    log_list = []
    # TODO：删除完全重复行，并记录日志
    step_name = "删除完全重复行"
    rule_desc = "移除整行完全一致的重复记录，重复样本无增量信息"
    before_cnt = df.shape[0]
    dup_num = df.duplicated().sum()
    df = df.drop_duplicates()
    after_cnt = df.shape[0]
    log_list.append({
        "处理步骤": step_name,
        "处理规则": rule_desc,
        "处理前记录数": before_cnt,
        "处理后记录数": after_cnt,
        "影响记录数": dup_num
    })
    # TODO：对 NUMERIC_MISSING_COLS 使用中位数填补，并记录每列影响数量
    for col in NUMERIC_MISSING_COLS:
        step_name = f"填充数值字段缺失：{col}"
        rule_desc = "使用全量总体中位数填充，不按Churn分组，避免目标泄露"
        before_cnt = df.shape[0]
        miss_num = df[col].isna().sum()
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        after_cnt = df.shape[0]
        log_list.append({
            "处理步骤": step_name,
            "处理规则": rule_desc,
            "处理前记录数": before_cnt,
            "处理后记录数": after_cnt,
            "影响记录数": miss_num
        })
    # TODO：对 CATEGORY_MAPPINGS 完成类别标准化，并记录每条映射影响数量
    for col, map_dict in CATEGORY_MAPPINGS.items():
        for old_val, new_val in map_dict.items():
            step_name = f"类别标准化 {col}: {old_val} → {new_val}"
            rule_desc = "同义业务文本统一为标准取值，方便分组统计"
            before_cnt = df.shape[0]
            change_num = (df[col] == old_val).sum()
            df[col] = df[col].replace(old_val, new_val)
            after_cnt = df.shape[0]
            log_list.append({
                "处理步骤": step_name,
                "处理规则": rule_desc,
                "处理前记录数": before_cnt,
                "处理后记录数": after_cnt,
                "影响记录数": change_num
            })
    # TODO：将 Churn 和 Complain 转为整数类型
    for col in ["Churn", "Complain"]:
        step_name = f"类型转换：{col} 转为整数"
        rule_desc = "流失/投诉二分类标签统一为整数类型，适配建模"
        before_cnt = df.shape[0]
        df[col] = df[col].astype(int)
        after_cnt = df.shape[0]
        log_list.append({
            "处理步骤": step_name,
            "处理规则": rule_desc,
            "处理前记录数": before_cnt,
            "处理后记录数": after_cnt,
            "影响记录数": 0
        })
    # TODO：返回 cleaned_df 与 cleaning_log
    cleaning_log = pd.DataFrame(log_list)
    return df, cleaning_log

### 任务 3：运行清洗函数并查看日志

In [16]:
# TODO：执行清洗
cleaned_df, cleaning_log = clean_ecommerce_data(raw_df)

display(cleaning_log)
cleaned_df.head()

,处理步骤,处理规则,处理前记录数,处理后记录数,影响记录数
0,删除完全重复行,移除整行完全一致的重复记录，重复样本无增量信息,5630,5630,0
1,填充数值字段缺失：Tenure,使用全量总体中位数填充，不按Churn分组，避免目标泄露,5630,5630,264
2,填充数值字段缺失：WarehouseToHome,使用全量总体中位数填充，不按Churn分组，避免目标泄露,5630,5630,251
3,填充数值字段缺失：HourSpendOnApp,使用全量总体中位数填充，不按Churn分组，避免目标泄露,5630,5630,255
4,填充数值字段缺失：OrderAmountHikeFromlastYear,使用全量总体中位数填充，不按Churn分组，避免目标泄露,5630,5630,265
5,填充数值字段缺失：CouponUsed,使用全量总体中位数填充，不按Churn分组，避免目标泄露,5630,5630,256
6,填充数值字段缺失：OrderCount,使用全量总体中位数填充，不按Churn分组，避免目标泄露,5630,5630,258
7,填充数值字段缺失：DaySinceLastOrder,使用全量总体中位数填充，不按Churn分组，避免目标泄露,5630,5630,307
8,类别标准化 PreferredLoginDevice: Phone → Mobile Phone,同义业务文本统一为标准取值，方便分组统计,5630,5630,1231
9,类别标准化 PreferredPaymentMode: COD → Cash on Deli...,同义业务文本统一为标准取值，方便分组统计,5630,5630,365


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93
1,50002,1,9.00,Mobile Phone,1,8.00,UPI,Male,3.00,4,Mobile Phone,3,Single,7,1,15.00,0.00,1.00,0.00,120.90
2,50003,1,9.00,Mobile Phone,1,30.00,Debit Card,Male,2.00,4,Mobile Phone,3,Single,6,1,14.00,0.00,1.00,3.00,120.28
3,50004,1,0.00,Mobile Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07
4,50005,1,0.00,Mobile Phone,1,12.00,Credit Card,Male,3.00,3,Mobile Phone,5,Single,3,0,11.00,1.00,1.00,3.00,129.60


---
## 5. 数据转换与候选异常值检查

为便于第五天分析，请新增：

- TenureGroup：用户使用时长分层；
- IsMobileLogin：是否主要使用移动端登录；
- 候选异常值报告：WarehouseToHome、OrderCount、CashbackAmount。

候选异常值只记录，不在本项目中自动删除。

In [17]:
def iqr_outlier_summary(series):
    """输出 IQR 候选异常值摘要。"""
    series = series.dropna()
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    return {
        "Q1": q1,
        "Q3": q3,
        "下限": lower,
        "上限": upper,
        "候选异常值数量": int(((series < lower) | (series > upper)).sum())
    }
# TODO：构建 tenure_bins、tenure_labels，并用 pd.cut 新建 TenureGroup
# TODO：新建 IsMobileLogin，移动端为 1，其他设备为 0
# TODO：生成 outlier_report（每行对应一个待检查字段）

# 1. 构建Tenure分层区间，新建TenureGroup
tenure_bins = [0, 12, 24, 36, 48, 60, float("inf")]
tenure_labels = ["0-12个月", "12-24个月", "24-36个月", "36-48个月", "48-60个月", "60个月以上"]
cleaned_df["TenureGroup"] = pd.cut(cleaned_df["Tenure"], bins=tenure_bins, labels=tenure_labels, right=False)

# 2. 新建IsMobileLogin：移动端登录为1，其余0
mobile_dev = ["Mobile Phone"]
cleaned_df["IsMobileLogin"] = cleaned_df["PreferredLoginDevice"].apply(lambda x: 1 if x in mobile_dev else 0)

# 3. 生成候选异常值报告（WarehouseToHome、OrderCount、CashbackAmount）
outlier_cols = ["WarehouseToHome", "OrderCount", "CashbackAmount"]
outlier_report = []
for col in outlier_cols:
    res = iqr_outlier_summary(cleaned_df[col])
    res["字段名"] = col
    outlier_report.append(res)
outlier_report = pd.DataFrame(outlier_report)
display(outlier_report)

# TODO：构建 tenure_bins、tenure_labels，并用 pd.cut 新建 TenureGroup
# TODO：新建 IsMobileLogin，移动端为 1，其他设备为 0
# TODO：生成 outlier_report（每行对应一个待检查字段）

,Q1,Q3,下限,上限,候选异常值数量,字段名
0,9.00,20.00,-7.50,36.50,2,WarehouseToHome
1,1.00,3.00,-2.00,6.00,703,OrderCount
2,145.77,196.39,69.84,272.33,438,CashbackAmount


### 任务 4：业务规则检查

统计以下不合规记录数，并写出你的处理结论：

- 使用时长小于 0；
- 仓库距离小于 0；
- 订单数小于或等于 0；
- 返现金额小于 0。

如果结果为 0，也应在项目日志或总结中记录。

In [18]:
# TODO：完成业务规则检查
# 逐条统计不合规数量
rule1_cnt = (cleaned_df["Tenure"] < 0).sum()          # 使用时长小于0
rule2_cnt = (cleaned_df["WarehouseToHome"] < 0).sum()# 仓库距离小于0
rule3_cnt = (cleaned_df["OrderCount"] <= 0).sum()    # 订单数≤0
rule4_cnt = (cleaned_df["CashbackAmount"] < 0).sum() # 返现金额小于0

business_rule_report = pd.DataFrame({
    "规则": [
        "使用时长(Tenure)小于0",
        "仓库距离(WarehouseToHome)小于0",
        "订单数量(OrderCount)小于或等于0",
        "返现金额(CashbackAmount)小于0"
    ],
    "不合规记录数": [rule1_cnt, rule2_cnt, rule3_cnt, rule4_cnt]
})
display(business_rule_report)

# 处理结论：
"""
1. 统计出的负数/零订单属于业务逻辑非法脏数据，无真实业务含义；
2. 本项目仅做记录留存，不直接删除样本，在后续建模分析阶段可根据模型需求过滤；
3. 所有不合规条数均写入项目报告，若条数为0也需留存记录，证明完成全量校验；
4. 若数量占比极低，可在建模时剔除；若占比高，需回溯原始数据源排查采集逻辑错误。
"""

,规则,不合规记录数
0,使用时长(Tenure)小于0,0
1,仓库距离(WarehouseToHome)小于0,0
2,订单数量(OrderCount)小于或等于0,0
3,返现金额(CashbackAmount)小于0,0


'\n1. 统计出的负数/零订单属于业务逻辑非法脏数据，无真实业务含义；\n2. 本项目仅做记录留存，不直接删除样本，在后续建模分析阶段可根据模型需求过滤；\n3. 所有不合规条数均写入项目报告，若条数为0也需留存记录，证明完成全量校验；\n4. 若数量占比极低，可在建模时剔除；若占比高，需回溯原始数据源排查采集逻辑错误。\n'

---
## 6. 项目验收与交付

请生成清洗后质量报告，比较清洗前后缺失值，并导出全部交付物。

In [19]:
# TODO：完成最终验收
quality_after = build_quality_report(cleaned_df)
assert cleaned_df[NUMERIC_MISSING_COLS].isna().sum().sum() == 0
assert "Phone" not in cleaned_df["PreferredLoginDevice"].unique()
assert "COD" not in cleaned_df["PreferredPaymentMode"].unique()
assert "CC" not in cleaned_df["PreferredPaymentMode"].unique()
assert {"TenureGroup", "IsMobileLogin"}.issubset(cleaned_df.columns)

# TODO：导出下列文件，使用 utf-8-sig 编码：
# 清洗前质量报告 关闭index
quality_before.to_csv(OUTPUT_DIR / "data_quality_before.csv", index=False, encoding="utf-8-sig")
# 清洗后质量报告
quality_after.to_csv(OUTPUT_DIR / "data_quality_after.csv", index=False, encoding="utf-8-sig")
# 清洗日志
cleaning_log.to_csv(OUTPUT_DIR / "cleaning_log.csv", index=False, encoding="utf-8-sig")
# 清洗后主数据表
cleaned_df.to_csv(OUTPUT_DIR / "ecommerce_customer_cleaned.csv", index=False, encoding="utf-8-sig")

# TODO：输出 outlier_report 和 business_rule_report
outlier_report.to_csv(OUTPUT_DIR / "outlier_report.csv", index=False, encoding="utf-8-sig")
business_rule_report.to_csv(OUTPUT_DIR / "business_rule_report.csv", index=False, encoding="utf-8-sig")

# TODO：输出交付文件的路径
print("原始数据文件路径：", DATA_PATH)
print("清洗前质量报告：", OUTPUT_DIR / "data_quality_before.csv")
print("清洗后质量报告：", OUTPUT_DIR / "data_quality_after.csv")
print("清洗日志：", OUTPUT_DIR / "cleaning_log.csv")
print("清洗后用户数据表：", OUTPUT_DIR / "ecommerce_customer_cleaned.csv")
print("异常值报告：", OUTPUT_DIR / "outlier_report.csv")
print("业务规则校验报告：", OUTPUT_DIR / "business_rule_report.csv")

display(quality_after)
print("数据清洗交付物全部导出完成，检查点通过")

原始数据文件路径： C:\Users\Lenovo\PyCharmMiscProject\ecommerce-user-analysis-24012399\data\E Commerce Dataset.xlsx
清洗前质量报告： C:\Users\Lenovo\PyCharmMiscProject\ecommerce-user-analysis-24012399\output\day04_project\data_quality_before.csv
清洗后质量报告： C:\Users\Lenovo\PyCharmMiscProject\ecommerce-user-analysis-24012399\output\day04_project\data_quality_after.csv
清洗日志： C:\Users\Lenovo\PyCharmMiscProject\ecommerce-user-analysis-24012399\output\day04_project\cleaning_log.csv
清洗后用户数据表： C:\Users\Lenovo\PyCharmMiscProject\ecommerce-user-analysis-24012399\output\day04_project\ecommerce_customer_cleaned.csv
异常值报告： C:\Users\Lenovo\PyCharmMiscProject\ecommerce-user-analysis-24012399\output\day04_project\outlier_report.csv
业务规则校验报告： C:\Users\Lenovo\PyCharmMiscProject\ecommerce-user-analysis-24012399\output\day04_project\business_rule_report.csv


,字段类型,缺失数量,缺失比例(%),唯一值数量
CustomerID,int64,0,0.00,5630
Churn,int64,0,0.00,2
Tenure,float64,0,0.00,36
PreferredLoginDevice,str,0,0.00,2
CityTier,int64,0,0.00,3
WarehouseToHome,float64,0,0.00,34
PreferredPaymentMode,str,0,0.00,5
Gender,str,0,0.00,2
HourSpendOnApp,float64,0,0.00,6
NumberOfDeviceRegistered,int64,0,0.00,6


数据清洗交付物全部导出完成，检查点通过


## 项目复盘

请在提交前用不超过 200 字回答：

1. 本项目发现了哪些数据质量问题？
2. 你对缺失值、类别不一致、候选异常值分别采取了什么策略？
3. 为什么清洗后的数据可以作为第五天分析的输入？
4. 哪些处理规则仍需要业务人员确认？

## 答:
1. 数据质量问题：多数值字段存在缺失；类别字段存在同义不一致写法；部分字段存在IQR候选异常值；存在完全重复行；部分字段存在业务非法负值。
2. 处理策略：缺失值用总体中位数填充；类别按业务规范统一标准化；候选异常值仅记录不自动删除；直接删除完全重复行。
3. 适配后续分析原因：数据已完成去重、缺失补齐、类别规整、新增分层特征，格式规范、质量稳定，不会干扰后续建模分析。
4. 需业务确认的规则：异常值是否剔除、业务负值的产生原因与处理方式、分层区间划分标准，都需要业务人员核验确认。

## GitHub提交检查

- [ ] Notebook已重启内核并从头运行成功；
- [ ] `output/day04_project/`包含清洗后数据、质量报告、清洗日志和异常/业务规则报告；
- [ ] 原始Excel没有被覆盖；
- [ ] 清洗函数、处理日志和项目复盘均已完成；
- [ ] 已提交并推送到个人GitHub仓库。